In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np

In [2]:
# Загрузка (используем относительный путь)
purchases = pd.read_csv('../data/raw/apparel-purchases.csv', parse_dates=['date'])
messages = pd.read_csv('../data/raw/apparel-messages.csv', parse_dates=['date'])
target = pd.read_csv('../data/raw/apparel-target_binary.csv')

Начнем с Изучения данных (EDA). Первичное знакомство

In [3]:
# 1.1. Общий обзор
datasets = {'Purchases': purchases, 'Messages': messages, 'Target': target}

for name, df in datasets.items():
    print(f"--- {name} ---")
    print(df.info())
    print(f"Duplicates: {df.duplicated().sum()}")
    print("\n")

--- Purchases ---
<class 'pandas.DataFrame'>
RangeIndex: 202208 entries, 0 to 202207
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   client_id     202208 non-null  int64         
 1   quantity      202208 non-null  int64         
 2   price         202208 non-null  float64       
 3   category_ids  202208 non-null  str           
 4   date          202208 non-null  datetime64[us]
 5   message_id    202208 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(2), str(2)
memory usage: 9.3 MB
None
Duplicates: 73020


--- Messages ---
<class 'pandas.DataFrame'>
RangeIndex: 12739798 entries, 0 to 12739797
Data columns (total 7 columns):
 #   Column            Dtype         
---  ------            -----         
 0   bulk_campaign_id  int64         
 1   client_id         int64         
 2   message_id        str           
 3   event             str           
 4   channel           st

In [4]:
# 1.2. Проверка баланса классов
print("Target distribution:")
print(target['target'].value_counts(normalize=True))

Target distribution:
target
0    0.980722
1    0.019278
Name: proportion, dtype: float64


In [5]:
# 1.3. Временные рамки
print(f"Purchases period: {purchases['date'].min()} to {purchases['date'].max()}")
print(f"Messages period: {messages['date'].min()} to {messages['date'].max()}")

Purchases period: 2022-05-16 00:00:00 to 2024-02-16 00:00:00
Messages period: 2022-05-19 00:00:00 to 2024-02-15 00:00:00


По результатам первичного знакомства: 
1.9% целевого класса! Это экстремальный дисбаланс. Чтобы модель не просто выдавала «ноль» для всех подряд, нам придется приложить усилия при проектировании признаков и настройке весов.

Также у нас 73 020 дубликатов в покупках — это почти 36% данных. Нужно их убрать, так как это, скорее всего, техническая ошибка записи.

Переходим к следующему этапу. Разработка признаков (Feature Engineering)

In [7]:
# 2.1. Очистка данных
purchases = purchases.drop_duplicates()

# Опорная дата для расчета Recency (на день позже последней покупки в логах)
snapshot_date = purchases['date'].max() + pd.Timedelta(days=1)

# 2.2. Агрегация покупок (RFM-метрики)
client_purchases = purchases.groupby('client_id').agg({
    'date': [
        lambda x: (snapshot_date - x.max()).days, # Recency (дней с последней покупки)
        lambda x: (snapshot_date - x.min()).days, # Tenure (дней с первой покупки)
        'count'                                   # Frequency (общее кол-во заказов)
    ],
    'price': ['sum', 'mean', 'max'],              # Monetary (траты)
    'quantity': ['sum', 'mean']                   # Объем заказов
})

# Сглаживаем названия колонок
client_purchases.columns = ['recency', 'tenure', 'frequency', 'total_spend', 'avg_check', 'max_item_price', 'total_quantity', 'avg_quantity']
client_purchases = client_purchases.reset_index()

# 2.3. Маркетинговые признаки (из apparel-messages)
# Считаем количество разных событий для каждого клиента
message_stats = messages.groupby(['client_id', 'event']).size().unstack(fill_value=0).reset_index()

# Переименуем колонки для ясности (например, 'opened' -> 'm_opened')
message_stats.columns = ['client_id'] + [f'm_{col}' for col in message_stats.columns if col != 'client_id']

# Добавим CTR (Click-through rate) - если есть клики и открытия
if 'm_opened' in message_stats.columns and 'm_delivered' in message_stats.columns:
    message_stats['m_open_rate'] = message_stats['m_opened'] / (message_stats['m_delivered'] + 1)

# 2.4. Сборка финального датасета
df_full = target.merge(client_purchases, on='client_id', how='left')
df_full = df_full.merge(message_stats, on='client_id', how='left')

# Заполняем пропуски (те, кто ничего не покупал или не получал писем)
df_full = df_full.fillna(0)

print(f"Shape of final dataset: {df_full.shape}")
df_full.head()

Shape of final dataset: (49849, 21)


,client_id,target,recency,tenure,frequency,total_spend,avg_check,max_item_price,total_quantity,avg_quantity,...,m_close,m_complain,m_hard_bounce,m_hbq_spam,m_open,m_purchase,m_send,m_soft_bounce,m_subscribe,m_unsubscribe
0,1515915625468060902,0,631,631,5,4795.0,959.000000,1999.0,5,1.0,...,0.0,0.0,0.0,0.0,35.0,5.0,126.0,0.0,0.0,1.0
1,1515915625468061003,1,409,409,6,14135.0,2355.833333,3499.0,6,1.0,...,0.0,0.0,0.0,0.0,5.0,1.0,154.0,0.0,0.0,0.0
2,1515915625468061099,0,641,641,1,299.0,299.000000,299.0,1,1.0,...,0.0,0.0,2.0,0.0,51.0,0.0,215.0,0.0,0.0,0.0
3,1515915625468061100,0,7,7,1,1049.0,1049.000000,1049.0,1,1.0,...,0.0,0.0,1.0,0.0,163.0,1.0,267.0,1.0,0.0,0.0
4,1515915625468061170,0,245,328,8,14102.0,1762.750000,2699.0,8,1.0,...,0.0,0.0,0.0,0.0,31.0,3.0,243.0,0.0,0.0,0.0
